In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
! pip install -q transformers datasets peft accelerate trl bitsandbytes
! pip install -q transformers datasets peft accelerate torch
! pip install --upgrade "torchao>=0.16.0"

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from tqdm import tqdm
from transformers import get_scheduler
from peft import LoraConfig
from trl import SFTTrainer
from huggingface_hub import login
from torch.utils.data import DataLoader
import time

  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
! pip install datasets


In [ ]:
from datasets import load_dataset

# Using xlangai/spider as it is a widely accessible version on the Hub
Dataset = load_dataset("xlangai/spider")
print(Dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/5.51k [00:00<?, ?B/s]

spider/train-00000-of-00001.parquet:   0%|          | 0.00/831k [00:00<?, ?B/s]

spider/validation-00000-of-00001.parquet:   0%|          | 0.00/126k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1034 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 7000
    })
    validation: Dataset({
        features: ['db_id', 'query', 'question', 'query_toks', 'query_toks_no_value', 'question_toks'],
        num_rows: 1034
    })
})


In [ ]:
from huggingface_hub import login
login()

In [ ]:
# ----------------------------
# Step 1: Hugging Face Login
# ----------------------------
HF_TOKEN = ""   # replace with your token
login(HF_TOKEN)

# ----------------------------
# Step 2: Load Tokenizer & Model
# ----------------------------
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    # Qwen2.5 base tokenizer has no dedicated pad token by default.
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,   # 8-bit is sufficient for a 3B model on a T4/free-tier GPU
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config= bnb_config,  # memory efficient, works on small GPU
    device_map="auto",
    token=HF_TOKEN
)

# ----------------------------
# Step 3: LoRA Config
# ----------------------------
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["embed_tokens",
    "q_proj",
    "v_proj",
    "o_proj",
    "k_proj",
    "up_proj",
    "down_proj",
    "gate_proj",],  # key attention layers
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

# ----------------------------
# Step 4: Load Spider Dataset
# ----------------------------
# Updated to use the namespaced URI to avoid HfUriError
dataset = load_dataset("xlangai/spider")

# ----------------------------
# Step 5: Format Spider -> Prompt/Response
# ----------------------------
def format_prompt(example):
    # prompt: natural language -> SQL
    prompt = f"Translate the following question into SQL:\nQuestion: {example['question']}\nSQL:"
    return {
        "input_text": prompt,
        "target_text": example["query"]
    }

# --- Sample 60% of Spider train and validation sets for faster training ---
from datasets import load_dataset
dataset = load_dataset("xlangai/spider")
train_subset = dataset["train"].train_test_split(test_size=0.4, seed=42)["train"]
val_subset = dataset["validation"].train_test_split(test_size=0.4, seed=42)["train"]

# Use these subsets for prompt formatting and training
train_dataset = train_subset.map(format_prompt)
val_dataset = val_subset.map(format_prompt)

#train_dataset = dataset["train"].map(format_prompt)
#val_dataset = dataset["validation"].map(format_prompt)

# ----------------------------
# Step 6: Torch Dataset Class
# ----------------------------
from torch.utils.data import Dataset  # Import Dataset from torch.utils.data

class SpiderText2SQLDataset(Dataset):
    """

    Builds a single concatenated [prompt + target + eos] sequence per
    example. labels are a clone of input_ids with:
      - the prompt portion masked to -100 (the model is not trained to
        "predict" the prompt, only the SQL completion that follows it)
      - padding tokens masked to -100. This is the standard causal-LM SFT recipe.

    """

    def __init__(self, hf_dataset, tokenizer, max_len=512):
        self.dataset = hf_dataset
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        prompt = item["input_text"]
        target = item["target_text"]

        prompt_ids = self.tokenizer(prompt, add_special_tokens=False)["input_ids"]
        target_ids = self.tokenizer(target, add_special_tokens=False)["input_ids"]
        eos_id = self.tokenizer.eos_token_id

        full_ids = prompt_ids + target_ids + [eos_id]

        # Truncate from the LEFT if too long, so the SQL target (the part
        # that actually matters for the loss) is never cut off.
        if len(full_ids) > self.max_len:
            full_ids = full_ids[-self.max_len:]
            prompt_len = max(0, len(full_ids) - (len(target_ids) + 1))
        else:
            prompt_len = len(prompt_ids)

        pad_len = self.max_len - len(full_ids)
        input_ids = full_ids + [self.tokenizer.pad_token_id] * pad_len
        attention_mask = [1] * len(full_ids) + [0] * pad_len

        labels = list(input_ids)
        for i in range(min(prompt_len, len(labels))):
            labels[i] = -100               # mask prompt tokens
        for i in range(len(full_ids), len(labels)):
            labels[i] = -100               # mask padding tokens

        return {
            "input_ids": torch.tensor(input_ids),
            "attention_mask": torch.tensor(attention_mask),
            "labels": torch.tensor(labels),
        }

train_torch = SpiderText2SQLDataset(train_dataset, tokenizer)
val_torch = SpiderText2SQLDataset(val_dataset, tokenizer)

train_loader = DataLoader(train_torch, batch_size=4, shuffle=True)
val_loader = DataLoader(val_torch, batch_size=4)

# ----------------------------
# Step 7: Optimizer & Scheduler
# ----------------------------
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
num_training_steps = len(train_loader) * 1  # 1 epoch for demo
lr_scheduler = get_scheduler(
    "linear", optimizer=optimizer,
    num_warmup_steps=100,
    num_training_steps=num_training_steps
)

# ----------------------------
# Step 8: Training Loop
# ----------------------------
import os
import json as _json

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.train()

CHECKPOINT_ROOT = "./checkpoints"
CHECKPOINT_EVERY = 500       # matches your remembered convention: 500_chk, 1000_chk, 1500_chk ...
LOG_FLUSH_EVERY = 50         # flush loss log to disk this often so a disconnect loses little
os.makedirs(CHECKPOINT_ROOT, exist_ok=True)

# Google Drive backup for the loss log (Drive is mounted in the cell above).
# Only the JSON log is backed up here -- small and cheap at every checkpoint.
# NOTE: the model checkpoint folders themselves (./checkpoints/*_chk) are NOT
# backed up by this -- they only live on the ephemeral Colab disk. If you
# also want the LoRA weights backed up over a long 3-epoch run, uncomment
# the shutil.copytree() block inside save_checkpoint below.
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/qwen_finetuning_backups"
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

loss_history = []   # list of {"step", "epoch", "batch", "loss"} -- this is the real training log
step = 0

def save_checkpoint(tag: str):
    chk_dir = f"{CHECKPOINT_ROOT}/{tag}"
    model.save_pretrained(chk_dir)
    tokenizer.save_pretrained(chk_dir)
    with open(f"{chk_dir}/loss_log.json", "w") as f:
        _json.dump(loss_history, f, indent=2)
    print(f"Checkpoint saved: {chk_dir}  (global step {step})")

    # Back up the loss log to Google Drive every time a checkpoint is saved,
    # so training evidence survives even if this Colab runtime disconnects.
    try:
        drive_latest = f"{DRIVE_BACKUP_DIR}/loss_log_latest.json"
        drive_snapshot = f"{DRIVE_BACKUP_DIR}/loss_log_{tag}.json"
        with open(drive_latest, "w") as f:
            _json.dump(loss_history, f, indent=2)
        with open(drive_snapshot, "w") as f:
            _json.dump(loss_history, f, indent=2)
        print(f"Backed up loss log to Google Drive: {drive_snapshot}")
    except Exception as e:
        print(f"WARNING: Could not back up loss log to Google Drive: {e}")

    # OPTIONAL: uncomment to also back up the LoRA weights to Drive at each
    # checkpoint (recommended for a full 3-epoch run since the model files
    # only exist on the ephemeral Colab disk otherwise).
    # import shutil
    # drive_chk_dir = f"{DRIVE_BACKUP_DIR}/{tag}"
    # shutil.copytree(chk_dir, drive_chk_dir, dirs_exist_ok=True)
    # print(f"Backed up checkpoint weights to Google Drive: {drive_chk_dir}")

def flush_loss_log():
    with open(f"{CHECKPOINT_ROOT}/loss_log_latest.json", "w") as f:
        _json.dump(loss_history, f, indent=2)

num_epochs = 3
gradient_accumulation_steps = 4

try:
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")
        total_loss = 0
        epoch_start = time.time()
        total_steps = len(train_loader)

        for batch_idx, batch in enumerate(train_loader, start=1):
            step += 1

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss / gradient_accumulation_steps
            loss_value = loss.item() * gradient_accumulation_steps
            total_loss += loss_value

            loss.backward()

            if step % gradient_accumulation_steps == 0:
                optimizer.step()
                optimizer.zero_grad()
                lr_scheduler.step()

            loss_history.append({
                "step": step, "epoch": epoch + 1, "batch": batch_idx, "loss": loss_value
            })

            if step % LOG_FLUSH_EVERY == 0:
                flush_loss_log()

            # Checkpoint on a Global step counter 500_chk, 1000_chk, 1500_chk, etc
            if step % CHECKPOINT_EVERY == 0:
                save_checkpoint(f"{step}_chk")

            elapsed = time.time() - epoch_start
            avg_time_per_step = elapsed / batch_idx
            steps_left = total_steps - batch_idx
            eta_epoch = avg_time_per_step * steps_left

            print(f"Epoch {epoch+1} | Step {batch_idx}/{total_steps} | Global step {step} |"
                  f" Loss: {loss_value:.4f} | ETA (Epoch): {eta_epoch/60:.2f} min")

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch + 1} completed. Average Loss: {avg_loss:.4f}")

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                val_loss += outputs.loss.item()
        avg_val_loss = val_loss / len(val_loader)
        print(f"Validation Loss after Epoch {epoch + 1}: {avg_val_loss:.4f}")
        model.train()

except KeyboardInterrupt:
    print("\nTraining interrupted manually -- saving emergency checkpoint and log...")
    save_checkpoint(f"{step}_chk_interrupted")
    flush_loss_log()

# ----------------------------
# Step 9: Save Final LoRA Adapter
# ----------------------------
save_dir = "./qwen_spider_lora"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
flush_loss_log()

print(f"Model saved at {save_dir}")
print(f"Full loss log saved at {CHECKPOINT_ROOT}/loss_log_latest.json")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...


Epoch 1/3


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Epoch 1 | Step 1/1050 | Global step 1 | Loss: 3.0288 | ETA (Epoch): 140.82 min
Epoch 1 | Step 2/1050 | Global step 2 | Loss: 5.0706 | ETA (Epoch): 131.29 min
Epoch 1 | Step 3/1050 | Global step 3 | Loss: 4.3308 | ETA (Epoch): 128.95 min
Epoch 1 | Step 4/1050 | Global step 4 | Loss: 2.7215 | ETA (Epoch): 132.85 min
Epoch 1 | Step 5/1050 | Global step 5 | Loss: 1.4679 | ETA (Epoch): 134.07 min
Epoch 1 | Step 6/1050 | Global step 6 | Loss: 1.5585 | ETA (Epoch): 133.18 min
Epoch 1 | Step 7/1050 | Global step 7 | Loss: 1.4434 | ETA (Epoch): 132.99 min
Epoch 1 | Step 8/1050 | Global step 8 | Loss: 2.1034 | ETA (Epoch): 132.99 min
Epoch 1 | Step 9/1050 | Global step 9 | Loss: 3.7720 | ETA (Epoch): 133.04 min
Epoch 1 | Step 10/1050 | Global step 10 | Loss: 1.4844 | ETA (Epoch): 133.60 min
Epoch 1 | Step 11/1050 | Global step 11 | Loss: 1.8634 | ETA (Epoch): 134.13 min
Epoch 1 | Step 12/1050 | Global step 12 | Loss: 1.7823 | ETA (Epoch): 134.86 min
Epoch 1 | Step 13/1050 | Global step 13 | Loss

/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:356: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


Checkpoint saved: ./checkpoints/500_chk  (global step 500)
Backed up loss log to Google Drive: /content/drive/MyDrive/qwen_finetuning_backups/loss_log_500_chk.json
Epoch 1 | Step 500/1050 | Global step 500 | Loss: 0.4452 | ETA (Epoch): 76.70 min
Epoch 1 | Step 501/1050 | Global step 501 | Loss: 0.8193 | ETA (Epoch): 76.56 min
Epoch 1 | Step 502/1050 | Global step 502 | Loss: 0.4141 | ETA (Epoch): 76.42 min
Epoch 1 | Step 503/1050 | Global step 503 | Loss: 0.6454 | ETA (Epoch): 76.29 min
Epoch 1 | Step 504/1050 | Global step 504 | Loss: 0.6532 | ETA (Epoch): 76.15 min
Epoch 1 | Step 505/1050 | Global step 505 | Loss: 0.4631 | ETA (Epoch): 76.01 min
Epoch 1 | Step 506/1050 | Global step 506 | Loss: 0.3213 | ETA (Epoch): 75.87 min
Epoch 1 | Step 507/1050 | Global step 507 | Loss: 0.3765 | ETA (Epoch): 75.73 min
Epoch 1 | Step 508/1050 | Global step 508 | Loss: 0.5039 | ETA (Epoch): 75.58 min
Epoch 1 | Step 509/1050 | Global step 509 | Loss: 0.3493 | ETA (Epoch): 75.44 min
Epoch 1 | Step 5

In [ ]:
# ----------------------------
# Step 10: Plot and Save the Training Loss Curve
# ----------------------------
import json
import matplotlib.pyplot as plt

CHECKPOINT_ROOT = "./drive/MyDrive/qwen_finetuning_backups"

with open(f"{CHECKPOINT_ROOT}/loss_log_latest.json") as f:
    loss_log = json.load(f)

steps = [x["step"] for x in loss_log]
losses = [x["loss"] for x in loss_log]

plt.figure(figsize=(10, 5))
plt.plot(steps, losses, linewidth=1)
plt.xlabel("Global training step")
plt.ylabel("Training loss")
plt.title("Qwen2.5-3B-Instruct LoRA Fine-Tuning on Spider -- Training Loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CHECKPOINT_ROOT}/training_loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Saved loss curve to {CHECKPOINT_ROOT}/training_loss_curve.png")
print(f"Raw loss log ({len(loss_log)} steps) at {CHECKPOINT_ROOT}/loss_log_latest.json")

<Figure size 1000x500 with 1 Axes>

Saved loss curve to ./drive/MyDrive/qwen_finetuning_backups/training_loss_curve.png
Raw loss log (2000 steps) at ./drive/MyDrive/qwen_finetuning_backups/loss_log_latest.json


### Adding Schema to Prompts
To solve Spider correctly, the model needs to know the database structure. We should modify the formatting function to include the `db_id` (or better yet, the schema content if you load the `tables.json` from the Spider dataset).

In [ ]:
def format_prompt_with_context(example):
    # Note: For best results, you should join this with the 'tables'
    # information from the Spider dataset to provide actual column names.
    prompt = f"""<|im_start|>system
You are a helpful assistant that writes SQL queries based on a database schema.<|im_end|>
<|im_start|>user
Database ID: {example['db_id']}
Question: {example['question']}
SQL Query:<|im_end|>
<|im_start|>assistant
"""
    return {
        "input_text": prompt,
        "target_text": example["query"] + "<|im_end|>"
    }

In [ ]:
from peft import PeftModel
import torch
import os
import glob
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# --- Load base model in 4-bit for efficiency (works on 8GB GPU) ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model_name = "Qwen/Qwen2.5-3B-Instruct"

print("Loading base Qwen model in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

# --- Load LoRA fine-tuned adapters: prefer the final save, else fall back
#     to the most recent checkpoint (useful if a Colab disconnect cut the
#     run short before "./qwen_spider_lora" was written). ---
def find_adapter_dir():
    final_dir = "./qwen_spider_lora"
    if os.path.isdir(final_dir):
        return final_dir
    chk_dirs = glob.glob("./checkpoints/*_chk")
    if not chk_dirs:
        raise FileNotFoundError(
            "No adapter found. Expected './qwen_spider_lora' or a "
            "'./checkpoints/<step>_chk' directory from training."
        )
    chk_dirs.sort(key=lambda p: int(os.path.basename(p).split("_")[0]))
    return chk_dirs[-1]

adapter_dir = find_adapter_dir()
print(f"Loading fine-tuned LoRA adapters from: {adapter_dir}")
ft_model = PeftModel.from_pretrained(base_model, adapter_dir)

ft_model.eval()

# --- SQL Generation Function ---
def generate_sql(question: str, schema: str, max_new_tokens: int = 256):
    """
    Given a natural language question, generate SQL + natural language response.
    """
    prompt = f"""
You are an expert SQL assistant.
Convert the following user request into a valid PostgreSQL query,
and then explain the result in plain English.
You must ONLY use the provided schema to write queries.
If a table or column is not listed, you must NOT invent it.
Always return only the SQL query, nothing else.
You must strictly follow the schema provided.
Never invent tables or columns. Only return SQL wrapped in ```sql ... ``` blocks.


User question: {"What are the name and budget of departments with average instructor salary gretaer than overall average?"}

SQL Query:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            top_p=0.9,
            do_sample=False
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = decoded[len(prompt):].strip()

    return response


# --- Example Inference ---
#if __name__ == "__main__":


#user_q = "What are the names of all students enrolled in the Computer Science department?"
#print("\n[User]:", user_q)
#answer = generate_sql(user_q)
#print("\n[Model]:", answer)